# 🚀 Simulador Interactiu de Tir Parabòlic (Cinemàtica 2D)
### Unitat Didàctica: *Moviment i Forces a la Física de Secundària i Batxillerat*
**Autor:** Casimir Victòria — *Treball de Final de Màster (Física i Química)*  
**Llicència:** Creative Commons Zero 1.0 (CC0 - Domini Públic)  

---

## 🎯 1. Objectius Didàctics i Competencials (LOMLOE)
1. **Modelitzar el moviment bidimensional** descomponent-lo en dos moviments independents (Principi de superposició de Galileu): un MRU en l'eix horitzontal ($) i un MRUA en l'eix vertical ($).
2. **Deduir de manera simbòlica** l'equació de la trajectòria, el temps de vol, l'altura màxima i l'abast màxim.
3. **Superar concepcions alternatives:** Comprovar com l'angle de llançament afecta l'abast i l'altura, experimentant amb angles complementaris (ex: 0^\circ$ i 0^\circ$) i diferents camps gravitatoris planetaris.

---

## 🧮 2. Deducció Simbòlica de les Equacions del Moviment
Emprem el motor de **càlcul simbòlic (SymPy / SageMath)** per a deduir formalment les expressions analítiques sense realitzar càlculs manuals feixucs.

In [ ]:
import sympy as sp
from IPython.display import display, Math

# Definim les variables i paràmetres físics simbòlics
t, g, v0, theta, y0 = sp.symbols("t g v_0 theta y_0", positive=True)

# Components de la velocitat inicial
v0x = v0 * sp.cos(theta)
v0y = v0 * sp.sin(theta)

# Equacions de posició en funció del temps t
x_t = v0x * t
y_t = y0 + v0y * t - sp.Rational(1, 2) * g * t**2

print("📌 Equacions paramètriques del moviment:")
display(Math(f"x(t) = {sp.latex(x_t)}"))
display(Math(f"y(t) = {sp.latex(y_t)}"))

# 1. Temps de vol (quan y(t) = 0 amb y0 = 0)
eq_impacte = y_t.subs(y0, 0)
solucions_t = sp.solve(eq_impacte, t)
t_vol = solucions_t[1]  # La solució t > 0

# 2. Abast màxim horitzontal x_max
x_max = sp.simplify(x_t.subs(t, t_vol))

# 3. Altura màxima y_max (quan v_y = 0 -> t_pujada = t_vol / 2)
t_pujada = t_vol / 2
y_max = sp.simplify(y_t.subs([(y0, 0), (t, t_pujada)]))

print("
🎯 Resultats simbòlics deduïts analíticament:")
display(Math(f"t_{{\text{{vol}}}} = {sp.latex(t_vol)}"))
display(Math(f"x_{{\text{{màx}}}} = {sp.latex(x_max)}"))
display(Math(f"y_{{\text{{màx}}}} = {sp.latex(y_max)}"))

## 🎮 3. Simulador Interactiu amb Lliscadors Dinàmics ()
Ajusta els valors de la **velocitat inicial ($)**, l'**angle de llançament ($	heta$)**, l'**alçada inicial ($)** i selecciona el **camp gravitatori planetari** per a observar la trajectòria en temps real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

# Diccionari de gravetats en diferents cossos celestes (m/s^2)
GRAVETATS = {
    "🌍 Terra (9.81 m/s²)": 9.81,
    "🌕 Lluna (1.62 m/s²)": 1.62,
    "🔴 Mart (3.71 m/s²)": 3.71,
    "🪐 Júpiter (24.79 m/s²)": 24.79
}

def simular_tir(v0=25.0, angle_graus=45.0, y0=0.0, planeta="🌍 Terra (9.81 m/s²)"):
    g = GRAVETATS[planeta]
    theta_rad = np.radians(angle_graus)
    
    v0x = v0 * np.cos(theta_rad)
    v0y = v0 * np.sin(theta_rad)
    
    # Càlcul del temps de vol exacte mitjançant la fórmula quadràtica
    # -0.5 * g * t^2 + v0y * t + y0 = 0
    discriminant = v0y**2 + 2 * g * y0
    t_vol = (v0y + np.sqrt(discriminant)) / g
    
    # Temps d'altura màxima
    t_hmax = v0y / g
    h_max = y0 + (v0y**2) / (2 * g) if t_hmax > 0 else y0
    x_hmax = v0x * t_hmax if t_hmax > 0 else 0
    
    # Abast màxim
    x_impacte = v0x * t_vol
    
    # Vector de temps per a la gràfica
    t_vals = np.linspace(0, t_vol, 300)
    x_vals = v0x * t_vals
    y_vals = y0 + v0y * t_vals - 0.5 * g * t_vals**2
    
    # Creació de la figura
    plt.figure(figsize=(10, 6), dpi=100)
    plt.plot(x_vals, y_vals, "b-", lw=3, label="Trajectòria Parabòlica")
    
    # Punts clau
    plt.plot(0, y0, "go", markersize=8, label=f"Punt Inicial (0, {y0:.1f} m)")
    plt.plot(x_hmax, h_max, "r^", markersize=10, label=f"Altura Màx. ({x_hmax:.1f}, {h_max:.1f} m)")
    plt.plot(x_impacte, 0, "kx", markersize=10, mew=3, label=f"Impacte ({x_impacte:.1f}, 0.0 m)")
    
    # Dibuix del vector de velocitat inicial
    escala_vector = min(x_impacte, h_max) * 0.2 if x_impacte > 0 else 5
    plt.quiver(0, y0, v0x, v0y, angles="xy", scale_units="xy", scale=v0/escala_vector, color="purple", width=0.005, label=f"Vector v0 ({v0:.1f} m/s a {angle_graus:.0f}°)")
    
    # Format i límits
    plt.axhline(0, color="black", lw=1.5)
    plt.axvline(0, color="black", lw=1)
    plt.title(f"Simulació de Tir Parabòlic — {planeta}
v₀ = {v0:.1f} m/s | θ = {angle_graus:.1f}° | y₀ = {y0:.1f} m", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Distància Horitzontal x (m)", fontsize=11, fontweight="bold")
    plt.ylabel("Alçada Vertical y (m)", fontsize=11, fontweight="bold")
    plt.xlim(-max(5, x_impacte*0.05), max(10, x_impacte*1.1))
    plt.ylim(-max(2, h_max*0.05), max(10, h_max*1.2))
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="upper right", frameon=True, shadow=True)
    plt.show()
    
    # Taula de resultats numèrics
    print(f"📊 RESULTATS NUMÈRICS DE LA SIMULACIÓ:")
    print(f"  • Temps total de vol:        {t_vol:.2f} s")
    print(f"  • Alçada màxima assolida:    {h_max:.2f} m (als {t_hmax:.2f} s)")
    print(f"  • Abast horitzontal total:   {x_impacte:.2f} m")
    print(f"  • Component horitzontal v_x: {v0x:.2f} m/s (constant)")
    print(f"  • Component vertical v_y0:   {v0y:.2f} m/s")

# Lliscadors interactius
interact(
    simular_tir,
    v0=FloatSlider(min=5.0, max=80.0, step=1.0, value=25.0, description="v₀ (m/s):"),
    angle_graus=FloatSlider(min=0.0, max=90.0, step=1.0, value=45.0, description="Angle θ (°):"),
    y0=FloatSlider(min=0.0, max=50.0, step=1.0, value=0.0, description="Alçada y₀ (m):"),
    planeta=Dropdown(options=list(GRAVETATS.keys()), value="🌍 Terra (9.81 m/s²)", description="Gravetat:")
);

---

## 🔍 4. Guia d'Indagació per a l'Alumnat (IBSE & Concepcions Alternatives)

### 🧪 Activitat 1: Els Angles Complementaris
* **Repte:** Fixa  = 30	ext{ m/s}$ i  = 0	ext{ m}$. Compara l'abast obtingut amb $	heta = 30^\circ$ i amb $	heta = 60^\circ$.
* **Pregunta de reflexió:** Què observes en l'abast màxim? Per què coincideixen? Compara ara els seus temps de vol i alçades màximes. Per què no són iguals?

### 🪐 Activitat 2: La Ciència en Altres Mons
* **Repte:** Llança el projectil a la Lluna ( = 1.62	ext{ m/s}^2$) i a Júpiter ( = 24.79	ext{ m/s}^2$) amb els mateixos valors de $ i $	heta$.
* **Pregunta de reflexió:** Com afecta la gravetat al temps de vol i a l'abast? És una relació lineal o inversa?